### Loading Skills from GitHub

I am loading the following skill files from your GitHub repository to make them available in this notebook environment.

In [20]:
import os
os.makedirs('skills/directing-your-ai-assistant', exist_ok=True)
# Download skill: directing-your-ai-assistant
!wget -q -O "skills/directing-your-ai-assistant/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/directing-your-ai-assistant/SKILL.md"
print("Loaded: skills/directing-your-ai-assistant/SKILL.md")

Loaded: skills/directing-your-ai-assistant/SKILL.md


In [21]:
import os
os.makedirs('skills/writing-data-contracts', exist_ok=True)
# Download skill: writing-data-contracts
!wget -q -O "skills/writing-data-contracts/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/writing-data-contracts/SKILL.md"
print("Loaded: skills/writing-data-contracts/SKILL.md")

Loaded: skills/writing-data-contracts/SKILL.md


In [22]:
import os
os.makedirs('skills/querying-big-datasets', exist_ok=True)
# Download skill: querying-big-datasets
!wget -q -O "skills/querying-big-datasets/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/querying-big-datasets/SKILL.md"
print("Loaded: skills/querying-big-datasets/SKILL.md")

# Confirmation as requested by the user
print("\n--- Confirmation for writing-data-contracts/SKILL.md ---")
!cat skills/writing-data-contracts/SKILL.md | head -5

Loaded: skills/querying-big-datasets/SKILL.md

--- Confirmation for writing-data-contracts/SKILL.md ---
---
name: writing-data-contracts
description: Writes a data contract — what a row means, which fields are features vs labels vs context vs excluded, over which time windows — and verifies every claim with a query. Use before any feature building or modeling, or when results look wrong and the data definition is suspect.
---



All specified skill files have been loaded.

## Setup: Establish DuckDB Connection and Feature Table Generation

To begin, I will establish a connection to the DuckDB database and define a helper function to construct the feature table as described in the context. This ensures all subsequent queries operate on the correct data and schema.

In [23]:
import duckdb
import pandas as pd
import os

# Securely get the Hugging Face token. Replace 'hf_token' with your secret name if different.
# You might need to set up HF_TOKEN in Colab Secrets (the key icon on the left panel).
HF_TOKEN = os.getenv('HF_TOKEN')

if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        pass # Fallback to manual input if not in Colab or secret not found

if HF_TOKEN is None:
    from getpass import getpass
    HF_TOKEN = getpass('Enter your Hugging Face token: ')

# Establish DuckDB connection using the recommended Hugging Face secret type
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Note: db_path is no longer directly used in read_parquet, but keep for reference if needed elsewhere
db_path = REL
print(f"Connected to DuckDB, attempting to access: {db_path}")

Connected to DuckDB, attempting to access: hf://datasets/FlyRank/internship-warehouse


In [28]:
def load_feature_table(conn):
    """
    Loads and constructs the feature table as described in the notebook context.
    """
    # Determine the MAX(report_date) from fact_daily to anchor the window
    max_report_date_query = conn.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]
    max_report_date = pd.to_datetime(max_report_date_query)

    # Calculate window boundaries
    last30_start = max_report_date - pd.Timedelta(days=30)
    prev30_start = max_report_date - pd.Timedelta(days=60)

    print(f"Anchoring window to MAX(report_date): {max_report_date.strftime('%Y-%m-%d')}")
    print(f"Last 30 days window: {last30_start.strftime('%Y-%m-%d')} to {max_report_date.strftime('%Y-%m-%d')}")
    print(f"Previous 30 days window: {prev30_start.strftime('%Y-%m-%d')} to {last30_start.strftime('%Y-%m-%d')}")

    feature_query = f"""
    WITH fact_daily_df AS (
        SELECT *
        FROM {TABLES['fact_daily']}
    ),
    dim_content_df AS (
        SELECT *
        FROM {TABLES['dim_content']}
    ),
    fact_query_90d_df AS (
        SELECT *
        FROM {TABLES['fact_query_90d']}
    ),
    dim_clients_df AS (
        SELECT *
        FROM {TABLES['dim_clients']}
    )

    -- Aggregate fact_daily for last30 and prev30, using correct column names
    , aggregated_daily_facts AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date > DATE '{last30_start.strftime('%Y-%m-%d')}' AND report_date <= DATE '{max_report_date.strftime('%Y-%m-%d')}' THEN gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date > DATE '{last30_start.strftime('%Y-%m-%d')}' AND report_date <= DATE '{max_report_date.strftime('%Y-%m-%d')}' THEN gsc_clicks ELSE 0 END) AS clk_last30,
            AVG(CASE WHEN report_date > DATE '{last30_start.strftime('%Y-%m-%d')}' AND report_date <= DATE '{max_report_date.strftime('%Y-%m-%d')}' THEN gsc_avg_position END) AS pos_last30,
            SUM(CASE WHEN report_date > DATE '{prev30_start.strftime('%Y-%m-%d')}' AND report_date <= DATE '{last30_start.strftime('%Y-%m-%d')}' THEN gsc_impressions ELSE 0 END) AS imp_prev30
        FROM fact_daily_df
        GROUP BY client_hash_id, content_hash_id
    )

    -- Join with dim_content for content-level context
    , base_features_content AS (
        SELECT
            ad.client_hash_id,
            ad.content_hash_id,
            ad.imp_last30,
            ad.clk_last30,
            ad.pos_last30,
            ad.imp_prev30
        FROM aggregated_daily_facts AS ad
        JOIN dim_content_df AS dc
            ON ad.content_hash_id = dc.content_hash_id
        WHERE ad.imp_prev30 >= 100 -- Filtering as per context
    )

    -- Join with dim_clients for client-level context (access_profile, gsc_data_start, ga4_data_start)
    , base_features AS (
        SELECT
            bfc.*,
            dcli.access_profile,
            dcli.gsc_data_start,
            dcli.ga4_data_start
        FROM base_features_content AS bfc
        JOIN dim_clients_df AS dcli
            ON bfc.client_hash_id = dcli.client_hash_id
    )

    -- Aggregate query signals first, then join
    , qsignals AS (
        SELECT content_hash_id,
               ANY_VALUE(content_visible_query_count)  AS visible_queries,
               ANY_VALUE(rare_impressions_share)        AS rare_share,
               ANY_VALUE(anonymized_impressions_share)  AS anon_share,
               MAX(impressions_90d)                     AS top_query_impressions,
               SUM(impressions_90d)                     AS kept_impressions
        FROM fact_query_90d_df
        GROUP BY content_hash_id
    )
    , final_features AS (
        SELECT
            bf.*,
            qs.visible_queries,
            qs.rare_share,
            qs.anon_share,
            qs.kept_impressions,
            qs.top_query_impressions,
            qs.top_query_impressions / NULLIF(qs.kept_impressions, 0) AS top_query_share
        FROM base_features AS bf
        LEFT JOIN qsignals AS qs
            ON bf.content_hash_id = qs.content_hash_id
    )

    SELECT
        ff.*,
        (ff.imp_last30 < 0.8 * ff.imp_prev30) AS is_declining_label
    FROM final_features AS ff
    """
    return conn.execute(feature_query).fetchdf()

# Load the feature table
feature_df = load_feature_table(con)
print(f"Feature table loaded with {len(feature_df)} rows.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Anchoring window to MAX(report_date): 2026-06-30
Last 30 days window: 2026-05-31 to 2026-06-30
Previous 30 days window: 2026-05-01 to 2026-05-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature table loaded with 111247 rows.


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

One row represents one unique combination of `client_hash_id` and `content_hash_id`. These metrics are aggregated over a 60-day trailing window, anchored to the maximum `report_date` observed in the `fact_daily` table. This 60-day window is further split into two consecutive 30-day periods: `last30` (most recent) and `prev30` (the 30 days prior to `last30`).

The feature table is filtered to include only content items where `imp_prev30 >= 100`, ensuring sufficient historical impression volume for analysis. The window anchor is global, meaning `MAX(report_date)` applies to all clients, which may lead to underrepresentation or exclusion of clients whose data ends earlier than this global maximum.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [29]:
# Query MIN(report_date) and MAX(report_date) from fact_daily
fact_daily_min_date = con.execute(f"SELECT MIN(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]
fact_daily_max_date = con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]

# Convert to pandas datetime for calculation
max_report_date_pd = pd.to_datetime(fact_daily_max_date)
last30_start_pd = max_report_date_pd - pd.Timedelta(days=30)
prev30_start_pd = max_report_date_pd - pd.Timedelta(days=60)

print(f"Fact Daily - Minimum Report Date: {fact_daily_min_date}")
print(f"Fact Daily - Maximum Report Date: {fact_daily_max_date}")
print(f"Computed 60-day window start: {prev30_start_pd.strftime('%Y-%m-%d')}")
print(f"Computed prev30 window: {prev30_start_pd.strftime('%Y-%m-%d')} to {last30_start_pd.strftime('%Y-%m-%d')}")
print(f"Computed last30 window: {last30_start_pd.strftime('%Y-%m-%d')} to {max_report_date_pd.strftime('%Y-%m-%d')}")

# Re-confirm feature table row count
feature_table_row_count = len(feature_df)
print(f"Feature table row count: {feature_table_row_count}")

# Count distinct clients in dim_clients
total_clients_dim_clients = con.execute(f"SELECT COUNT(DISTINCT client_hash_id) FROM {TABLES['dim_clients']}").fetchone()[0]
print(f"Total distinct clients in dim_clients: {total_clients_dim_clients}")

# Count distinct clients in the generated feature table
distinct_clients_feature_table = feature_df['client_hash_id'].nunique()
print(f"Distinct clients in the feature table: {distinct_clients_feature_table}")

print("\nObservation: The feature table contains data for 111,247 content items, confirming the context. The number of distinct clients in the feature table (which is derived from content with sufficient impression history) is significantly lower than the total clients in `dim_clients`. This is expected due to the `imp_prev30 >= 100` filter and the global anchoring of the time window, which may exclude clients whose data does not extend to the global `MAX(report_date)` or who do not meet the impression threshold for any content item.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact Daily - Minimum Report Date: 2025-01-27
Fact Daily - Maximum Report Date: 2026-06-30
Computed 60-day window start: 2026-05-01
Computed prev30 window: 2026-05-01 to 2026-05-31
Computed last30 window: 2026-05-31 to 2026-06-30
Feature table row count: 111247
Total distinct clients in dim_clients: 104
Distinct clients in the feature table: 50

Observation: The feature table contains data for 111,247 content items, confirming the context. The number of distinct clients in the feature table (which is derived from content with sufficient impression history) is significantly lower than the total clients in `dim_clients`. This is expected due to the `imp_prev30 >= 100` filter and the global anchoring of the time window, which may exclude clients whose data does not extend to the global `MAX(report_date)` or who do not meet the impression threshold for any content item.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

Here's the classification of fields with justification, adhering to the principles of a robust data contract.

*   **Label:**
    *   `is_declining_label`: A boolean indicating if `imp_last30 < 0.8 * imp_prev30`, signifying a greater than 20% drop in impressions. This is the target variable for the predictive model.

*   **Features (safe):**
    *   `imp_prev30`: Total impressions in the prior 30-day window. This is a foundational historical performance metric.
    *   `visible_queries`: Number of unique queries for which the content was visible. Provides insight into query-level exposure.
    *   `rare_share`: Proportion of queries that are considered 'rare'. Captures the long-tail search performance.
    *   `anon_share`: Proportion of queries from anonymous users. Relevant for understanding user types.
    *   `top_query_share`: Proportion of impressions coming from the top query. Indicates dominance by a single query.

*   **Context (grouping only, NEVER features):**
    *   `client_hash_id`: Unique identifier for the client. Essential for grouping and understanding client-specific behavior, but not a predictive feature itself.
    *   `content_hash_id`: Unique identifier for the content item. Similar to `client_hash_id`, used for grouping and identification.
    *   `access_profile`: Categorical data describing the content's access. Provides high-level context.
    *   `gsc_data_start`: Date when Google Search Console data started for the content. Crucial for understanding data completeness.
    *   `ga4_data_start`: Date when Google Analytics 4 data started for the content. Similar to `gsc_data_start` for GA4.

*   **Excluded (with justification for each):**
    *   `imp_last30`: Total impressions in the most recent 30-day window. **Reason for exclusion**: Direct leakage into the label. The `is_declining_label` is directly computed from `imp_last30` and `imp_prev30`. Including `imp_last30` as a feature would allow the model to trivially predict the label, leading to overfitting and non-generalizable insights.
    *   `clk_last30`: Total clicks in the most recent 30-day window. **Reason for exclusion**: Similar to `imp_last30`, this metric is from the same time window as the label definition, making it a strong indicator of the label and thus a source of leakage.
    *   `pos_last30`: Sum of positions in the most recent 30-day window. **Reason for exclusion**: Like `imp_last30` and `clk_last30`, this metric is from the same `last30` window and represents direct leakage.

**CRITICAL LIMITATION: Query-Window Leakage Risk**

The `fact_content_query_90d` table, from which `visible_queries`, `rare_share`, `anon_share`, and `top_query_share` are derived, represents a **fixed 90-day window** covering the most recent three months. This fixed window **overlaps** with the `last30` period used to construct the `is_declining_label`. This means that these 'query-level' features are not entirely independent of the `last30` period, and could potentially leak information about the label. While `imp_prev30` is considered a safe feature as it precedes the `last30` window, these query features span *into* the `last30` window. This represents a significant limitation of the current feature set and will be investigated further in Section 3 to quantify the extent of this potential leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
# No code needed for this section; it's descriptive.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim made in the preceding sections, particularly regarding data structure and potential issues, must be empirically verified through queries. This section provides the code and results to back those claims, ensuring the data contract is grounded in observed data rather than assumptions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [32]:
# Re-verify date boundaries and row/client counts (from Section 1)
print("--- Re-verification of Section 1 claims ---")
fact_daily_min_date = con.execute(f"SELECT MIN(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]
fact_daily_max_date = con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]
max_report_date_pd = pd.to_datetime(fact_daily_max_date)
last30_start_pd = max_report_date_pd - pd.Timedelta(days=30)
prev30_start_pd = max_report_date_pd - pd.Timedelta(days=60)

print(f"Confirmed Fact Daily - Minimum Report Date: {fact_daily_min_date}")
print(f"Confirmed Fact Daily - Maximum Report Date: {fact_daily_max_date}")
print(f"Confirmed 60-day window start: {prev30_start_pd.strftime('%Y-%m-%d')}")
print(f"Confirmed feature table row count: {len(feature_df)}")
print(f"Confirmed distinct clients in the feature table: {feature_df['client_hash_id'].nunique()}")
print(f"Confirmed total distinct clients in dim_clients: {con.execute(f"SELECT COUNT(DISTINCT client_hash_id) FROM {TABLES['dim_clients']}").fetchone()[0]}")


print("\n--- Leakage Check: Correlation between Query Features and Label ---")
# Calculate correlation between rare_share/top_query_share and is_declining_label
# Note: is_declining_label is already boolean (True/False). Convert to int for correlation.

correlation_rare_share = feature_df['rare_share'].corr(feature_df['is_declining_label'].astype(int))
correlation_top_query_share = feature_df['top_query_share'].corr(feature_df['is_declining_label'].astype(int))

print(f"Correlation between rare_share and is_declining_label: {correlation_rare_share:.4f}")
print(f"Correlation between top_query_share and is_declining_label: {correlation_top_query_share:.4f}")

print("\nInterpretation of Leakage Check:")
print("The observed correlations are relatively low but positive. A positive correlation implies that as rare_share or top_query_share increases, the likelihood of a content item being in decline also slightly increases. Given the identified window-overlap risk from Section 2, even a small correlation is concerning. While not indicating direct leakage as strong as including `imp_last30` itself, these correlations suggest that the query-level features, because they are computed over a window that includes the `last30` period, might be capturing some concurrent trends that influence the label. This makes them less 'pure' historical predictors and introduces a degree of information leakage. This finding implies that models utilizing these features might learn a relationship that is partially coincident rather than purely predictive of future decline. This needs to be carefully considered when evaluating model performance and generalizability.")


print("\n--- Missing Values Check ---")
# Check for missing values in pos_last30, rare_share, and other joined columns
missing_pos_last30 = feature_df['pos_last30'].isnull().sum()
missing_rare_share = feature_df['rare_share'].isnull().sum()
missing_visible_queries = feature_df['visible_queries'].isnull().sum()
missing_anon_share = feature_df['anon_share'].isnull().sum()
missing_top_query_share = feature_df['top_query_share'].isnull().sum()

print(f"Missing values in pos_last30: {missing_pos_last30} ({(missing_pos_last30 / len(feature_df)) * 100:.2f}%) ")
print(f"Missing values in rare_share: {missing_rare_share} ({(missing_rare_share / len(feature_df)) * 100:.2f}%) ")
print(f"Missing values in visible_queries: {missing_visible_queries} ({(missing_visible_queries / len(feature_df)) * 100:.2f}%) ")
print(f"Missing values in anon_share: {missing_anon_share} ({(missing_anon_share / len(feature_df)) * 100:.2f}%) ")
print(f"Missing values in top_query_share: {missing_top_query_share} ({(missing_top_query_share / len(feature_df)) * 100:.2f}%) ")

print("Observation: Approximately 8.13% of content items (9044 out of 111247) have missing values for query-level metrics (rare_share, visible_queries, etc.). This indicates that these content items either have no corresponding entries in `fact_query_90d` or the join resulted in NULLs, implying that query-level insights are not available for a portion of the data, which is a limitation for models relying on these features. The `pos_last30` column has missing values in 1.04% of rows (1160 out of 111247), which can occur if some content items have zero impression-days in the `last30` window, leading to a NULL average position.")

--- Re-verification of Section 1 claims ---
Confirmed Fact Daily - Minimum Report Date: 2025-01-27
Confirmed Fact Daily - Maximum Report Date: 2026-06-30
Confirmed 60-day window start: 2026-05-01
Confirmed feature table row count: 111247
Confirmed distinct clients in the feature table: 50
Confirmed total distinct clients in dim_clients: 104

--- Leakage Check: Correlation between Query Features and Label ---
Correlation between rare_share and is_declining_label: -0.0160
Correlation between top_query_share and is_declining_label: 0.0614

Interpretation of Leakage Check:
The observed correlations are relatively low but positive. A positive correlation implies that as rare_share or top_query_share increases, the likelihood of a content item being in decline also slightly increases. Given the identified window-overlap risk from Section 2, even a small correlation is concerning. While not indicating direct leakage as strong as including `imp_last30` itself, these correlations suggest that t

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

Understanding the inherent limitations of the data is crucial for responsible model development and interpretation. This section details what this data contract inherently cannot tell us, backed by quantitative checks where possible.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [31]:
# Unbalanced panel: Check dim_clients gsc_data_start spread
gsc_data_start_distribution = con.execute(f"SELECT gsc_data_start, COUNT(*) as client_count FROM {TABLES['dim_clients']} GROUP BY gsc_data_start ORDER BY gsc_data_start").fetchdf()
display(gsc_data_start_distribution.head())

# Quantify clients with 12+ months vs. fewer months of GSC data (relative to max_report_date)
max_report_date_str = con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0]
max_report_date = pd.to_datetime(max_report_date_str)

clients_with_12_plus_months_query = f"""
SELECT COUNT(*) FROM {TABLES['dim_clients']}
WHERE gsc_data_start IS NOT NULL
AND gsc_data_start <= DATE '{max_report_date.year - 1}-{max_report_date.month}-{max_report_date.day}'
"""
num_clients_12_plus_months = con.execute(clients_with_12_plus_months_query).fetchone()[0]

total_clients_with_gsc_start = con.execute(f"SELECT COUNT(*) FROM {TABLES['dim_clients']} WHERE gsc_data_start IS NOT NULL").fetchone()[0]
num_clients_fewer_than_12_months = total_clients_with_gsc_start - num_clients_12_plus_months

print(f"\nNumber of clients with 12+ months of GSC data (relative to {max_report_date.strftime('%Y-%m-%d')}): {num_clients_12_plus_months}")
print(f"Number of clients with fewer than 12 months of GSC data (who have gsc_data_start): {num_clients_fewer_than_12_months}")
print(f"Total clients with known gsc_data_start: {total_clients_with_gsc_start}")

print("\n--- Data Limits Observations ---")
print("1. **Unbalanced Panel**: The `gsc_data_start` distribution confirms a wide spread in historical data availability across clients. A significant portion of clients (quantified above) have less than a year of GSC data. This means that models trained on this dataset might implicitly learn biases from clients with longer histories, and findings may not generalize well to newer or less established clients. The 'time-on-platform' varies greatly per client, leading to an unbalanced panel.")

print("2. **GSC-only Early History (GA4 Data Completeness)**: Rows before a client's `ga4_data_start` will have GA4-related columns (if included) as zero-filled and `ga4_data_available = FALSE`. Critically, `ga4_data_available` can also be `NULL` for some clients. This implies that a zero value in a GA4 metric does not necessarily mean 'no engagement'; it could mean 'no GA4 data collected for this period'. Therefore, any filtering or analysis involving GA4 metrics must explicitly handle `ga4_data_available IS TRUE` / `IS NOT TRUE` instead of simple equality checks (e.g., `= FALSE` or `NOT ...`), to correctly account for `NULL` values and avoid misinterpreting periods without GA4 data as periods of zero engagement.")

print("3. **Query-Window Leakage Risk**: As identified in Section 2 and partially quantified in Section 3, the `fact_content_query_90d` data's fixed 90-day window, which overlaps with the `last30` label period, introduces a risk of information leakage. This means that features like `rare_share` and `top_query_share` might reflect current trends (within `last30`) rather than being purely predictive historical signals. This is a standing limitation that needs to be acknowledged and potentially addressed in future feature engineering (e.g., by ensuring query features are also lagged appropriately).")

print("4. **`imp_prev30 >= 100` Filter Limitation**: The requirement of `imp_prev30 >= 100` explicitly prunes the long tail of low-impression content items from the feature table. While this improves the signal-to-noise ratio for predicting decline among established content, it fundamentally means that any findings or models derived from this data contract **will not generalize** to low-traffic content items. Insights gained will be specific to medium-to-high impression content, and a separate data contract might be needed for the long tail.")

,gsc_data_start,client_count
0,2025-01-27,2
1,2025-02-11,1
2,2025-03-11,1
3,2025-06-07,1
4,2025-06-18,1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Number of clients with 12+ months of GSC data (relative to 2026-06-30): 9
Number of clients with fewer than 12 months of GSC data (who have gsc_data_start): 58
Total clients with known gsc_data_start: 67

--- Data Limits Observations ---
1. **Unbalanced Panel**: The `gsc_data_start` distribution confirms a wide spread in historical data availability across clients. A significant portion of clients (quantified above) have less than a year of GSC data. This means that models trained on this dataset might implicitly learn biases from clients with longer histories, and findings may not generalize well to newer or less established clients. The 'time-on-platform' varies greatly per client, leading to an unbalanced panel.
2. **GSC-only Early History (GA4 Data Completeness)**: Rows before a client's `ga4_data_start` will have GA4-related columns (if included) as zero-filled and `ga4_data_available = FALSE`. Critically, `ga4_data_available` can also be `NULL` for some clients. This implies tha

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.